In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "drive/MyDrive/FYP/ALLAM-7B"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()



`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(64000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
   

In [ ]:
def generate_chat(model, tokenizer, user_content: str, max_new_tokens: int):
    messages = [{"role": "user", "content": user_content}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,      # deterministic
            temperature=None,     # ignored when do_sample=False
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )
    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [ ]:
import re,json

In [ ]:
def extract_first_json(text:str):
    #this function aims to find {...} from the raw generated text
    if text is None:
        return None
    text = text.strip()
    return json.loads(text)


##### Changed the Extract First Json because it couldnt track the {...} block because it was nested in here

# Evaluating Prompt Engineering Methods On ABSA Dev

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, json
from tqdm import tqdm

emb = SentenceTransformer("BAAI/bge-m3")

def embed(text: str):
    return emb.encode(text, normalize_embeddings=True)

def load_embeded_absa_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(tqdm(f)):
            r = json.loads(line)
            rows.append({
                "id": i,
                "text": r["text"],
                "labels": r["labels"],
                "text_embedding": embed(r["text"]),
            })
    return rows

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
import json
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows
absa_test_path  = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"
absa_test = load_jsonl(absa_test_path)

In [ ]:
absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"

absa_train = load_jsonl(absa_train_path)
absa_train_emb = load_embeded_absa_jsonl(absa_train_path)


1564it [00:30, 50.56it/s]


In [ ]:
ABSA_POLARITIES = {"positive", "negative", "neutral", "conflict"}

def get_allowed_categories(rows):
    cats = set()
    for r in rows:
        for op in r["labels"]:
            cats.add(op["category"])
    return cats
ALLOWED_CATEGORIES = get_allowed_categories(absa_train)

In [ ]:
def retrieve_topk(query, corpus, k=3):
    q = embed(query)
    scored = []
    for r in corpus:
        sim = float(np.dot(q, r["text_embedding"]))
        scored.append({**r, "similarity_score": sim})
    scored.sort(key=lambda x: x["similarity_score"], reverse=True)
    return scored[:k]

def retrieve_absa_demos(corpus_rows, query_text, shots):
    retrieved = retrieve_topk(query_text, corpus_rows, shots)
    demos = []
    for r in retrieved:
        demos.append({"text": r["text"], "labels": r["labels"]})
    return demos


In [ ]:

def format_absa_demos(demos):
    lines = ["أمثلة قد تكون مشابهة (Examples):"]
    for d in demos:
        lines.append(f'النص: {d["text"]}')
        lines.append("الإجابة: " + json.dumps({"labels": d["labels"]}, ensure_ascii=False))
        lines.append("")
    return "\n".join(lines).strip()

def build_absa_prompt_rag(text, allowed_categories, demos=None):
    cats_str = ", ".join(sorted(list(allowed_categories)))

    header = (
        "أنت نظام لاستخراج مشاعر الجوانب (Aspect-Based Sentiment) من مراجعات الفنادق باللغة العربية.\n\n"
        "المهمة:\n"
        "استخرج جميع الآراء (opinions) الموجودة في النص.\n"
        "كل رأي يجب أن يكون زوجاً من:\n"
        "- category: جانب محدد بصيغة مثل HOTEL#CLEANLINESS\n"
        "- polarity: واحدة من {positive, negative, neutral, conflict}\n\n"
        "قيود مهمة جداً:\n"
        f"- استخدم فقط الفئات التالية ولا تكتب أكثر من 8: {cats_str}\n"
        "صيغة الإخراج بالضبط:\n"
        '{"labels":[{"category":"...","polarity":"..."}, ...]}\n'
        "لا تكتب أي شيء آخر. أرجع JSON فقط.\n"
        "- إذا لم تجد أي آراء، أرجع: {\"labels\":[]}\n"
        "لا تخمّن. لا تضف أي رأي إلا إذا كان واضحًا ومذكورًا في النص.\n"
        "أخرج بحد أقصى 8 آراء. إذا زادت، اختر الأوضح فقط.\n"
    )

    demo_block = (format_absa_demos(demos) + "\n\n") if demos else ""
    return f"{header}\n{demo_block}النص:\n{text}\n"


In [ ]:
def format_absa_prediction(obj, allowed_categories):
    if not isinstance(obj, dict) or "labels" not in obj:
        return None

    labels = obj["labels"]
    if not isinstance(labels, list):
        return None

    allowed_set = set(allowed_categories)
    out = set()

    for label in labels:
        if not isinstance(label, dict):
            continue
        cat = label.get("category")
        pol = label.get("polarity")
        if isinstance(cat, str): cat = cat.strip()
        if isinstance(pol, str): pol = pol.strip()

        if cat in allowed_set and pol in ABSA_POLARITIES:
            out.add((cat, pol))

    return out

In [ ]:
def f1_micro(y_true, y_pred):
    tp = fp = fn = 0
    for true, pred in zip(y_true, y_pred):
        tp += len(true & pred)
        fp += len(pred - true)
        fn += len(true - pred)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2*precision*recall)/(precision+recall) if (precision+recall) else 0.0
    return precision, recall, f1

In [ ]:
from tqdm import tqdm

def eval_absa_rag(rows, allowed_categories, demo_corpus_emb, demos=3):
    bad = 0
    y_pred, y_true = [], []

    for r in tqdm(rows):
        demo_examples = retrieve_absa_demos(demo_corpus_emb, r["text"], demos) if demos else None

        prompt = build_absa_prompt_rag(r["text"], allowed_categories, demos=demo_examples)

        raw = generate_chat(model, tokenizer, prompt, max_new_tokens=1000)
        block = extract_first_json(raw)
        pred = format_absa_prediction(block, allowed_categories)

        if pred is None:
            bad += 1
            pred = set()

        true = {(x["category"], x["polarity"]) for x in r["labels"]}
        y_true.append(true)
        y_pred.append(pred)

        print("TEXT:", r["text"])
        print("RAW:", raw[:800])
        print("TRUE:", true)
        print("PRED:", pred)

        torch.cuda.empty_cache()

    p, rc, f1 = f1_micro(y_true, y_pred)
    return {"p": p, "r": rc, "micro_f1": f1, "invalid_json": bad, "n": len(rows)}


In [ ]:
results = eval_absa_rag(absa_test, ALLOWED_CATEGORIES, absa_train_emb, demos=3)

  0%|          | 1/452 [00:02<22:04,  2.94s/it]

TEXT: فندق حلو جدا جدا جميل وعائلي ومناسب ولكن المواقف غير مجانية قريب من البحر مرتب بصراحة لا يعاب عليه إلا مواقف السيارات
RAW: {"labels": [{"category": "LOCATION#GENERAL", "polarity": "positive"}, {"category": "HOTEL#GENERAL", "polarity": "positive"}, {"category": "FACILITIES#GENERAL", "polarity": "positive"}]}
TRUE: {('FACILITIES#PRICES', 'negative'), ('LOCATION#GENERAL', 'positive'), ('HOTEL#GENERAL', 'conflict')}
PRED: {('LOCATION#GENERAL', 'positive'), ('HOTEL#GENERAL', 'positive'), ('FACILITIES#GENERAL', 'positive')}


  0%|          | 2/452 [00:04<16:31,  2.20s/it]

TEXT: فندق اكثر من رائع استمتعت في إقامتي في هذا الفندق الأكثر من رائع وأود أن اشكر جميع العاملين فيه وبالأخص الموظفة /هند من الجنسية العربيه في مكتب الاستقبال والحجوزات لتعاملها الرائع مع وباذن الله سيكون حجري في نفس الفندق في المره القادمة.
RAW: {"labels": [{"category": "SERVICE#GENERAL", "polarity": "positive"}, {"category": "HOTEL#GENERAL", "polarity": "positive"}]}
TRUE: {('HOTEL#GENERAL', 'positive'), ('SERVICE#GENERAL', 'positive')}
PRED: {('HOTEL#GENERAL', 'positive'), ('SERVICE#GENERAL', 'positive')}


  1%|          | 3/452 [00:05<10:14,  1.37s/it]

...

TEXT: فندق غير قابل للمعيشه أقسم بالله انا اول مره انزل في فندق بالسوء ده .. انا كنت حاجز مع شركة سياحة ووصلت الفندق الساعه 10 اول ما وصلت فوجئت من منظر الفندق من بره و جوه .. الاستقبال ضيق جداً جداً و الأرض مليانه تراب و السجاجيد مش نضيفه و المسئول عن الاستقبال بيعاملك ولا كأنك شغال عنده .. و بعد اكتر من حاجه كده كلمنا شركة السياحه قولنالها مينفعش طبعاً ننزل في فندق بالمستوي ده .. لحد ما لقينا فندق تاني و حجزنا فيه و لغينا الحجز في الفندق ده .. و في الأخر أنصح الجميع محدش يروح الفندق ده !
RAW: {"labels": [{"category": "HOTEL#GENERAL", "polarity": "negative"}]}
TRUE: {('FACILITIES#DESIGN_FEATURES', 'negative'), ('HOTEL#GENERAL', 'negative'), ('HOTEL#DESIGN_FEATURES', 'negative'), ('HOTEL#CLEANLINESS', 'negative'), ('HOTEL#QUALITY', 'negative'), ('SERVICE#GENERAL', 'negative')}
PRED: {('HOTEL#GENERAL', 'negative')}


100%|█████████▉| 451/452 [21:06<00:01,  1.52s/it]

TEXT: غرف سيئة وخدمة سيئة - بصورة عامة مخيب للآمال الموقع كان جيداً، على بعد بناية واحدة إلى الخلف من شاطئ كوباكابانا وعلى مقربة من ايبانيما. على الرغم من ذلك، فقد كان الفندق مخيباً للآمال بصفة عامة:- الموظفون كانوا غير لطيفين على الرغم من أنهم يتحدثون اللغة الإنجليزية. - لم يوفّر الفندق سريراً مزدوجاً عندما وصلنا. - عندما تم صنع سرير مزدوج (ضم سريرين مفردين معاً) لم تكن ملاءات السرير مناسبة. - الإنترنت كان معطلاً في أحد الأيام. - منطقة تناول الطعام غالباً ما تكون مزدحمة جداً للحصول على مقعد للإفطار
RAW: {"labels": [{"category": "ROOMS#GENERAL", "polarity": "negative"}, {"category": "SERVICE#GENERAL", "polarity": "negative"}]}
TRUE: {('ROOMS#GENERAL', 'negative'), ('ROOMS_AMENITIES#GENERAL', 'negative'), ('FACILITIES#MISCELLANEOUS', 'negative'), ('ROOMS_AMENITIES#DESIGN_FEATURES', 'negative'), ('HOTEL#GENERAL', 'negative'), ('SERVICE#GENERAL', 'negative'), ('FACILITIES#GENERAL', 'negative'), ('LOCATION#GENERAL', 'positive')}
PRED: {('ROOMS#GENERAL', 'negative'), ('SERVICE#GENERAL', 'ne

100%|██████████| 452/452 [21:12<00:00,  2.82s/it]

TEXT: ملعب الفران و مزبله الغردقه اسواء مكان ممكن تزوره اوتقيم فيه لانه باختصار ملعب فئران وحشرات طبعا انتى عمرك ما تلاحظ ده الا لما تدخل الاماكن المستخبيه وانا يا جماعه ما انصحش حد بيه .... حتى العلملين اشكالهم سيئه جدا مش عارف جايبنهم منين وما بيهتموش بحاجه ويعاملوك بقرف لان يا جماعه مرتباتهم ضعيفه جدا تخيلوا عامل بياخد 350 جنيه مصرى فقط لاغير فندق محترم زى ده يدفع المبلغ ده لعامل طب ازاى العامل يشتغل بنفس وحتى العامل اللى حيرضى بالوضع ده اكيد عامل قليل الخبره او منعدمها يحسسك انك قاعد فى قهوه بلدى
RAW: {"labels": [{"category": "LOCATION#GENERAL", "polarity": "negative"}, {"category": "HOTEL#GENERAL", "polarity": "negative"}, {"category": "SERVICE#GENERAL", "polarity": "negative"}, {"category": "ROOMS#CLEANLINESS", "polarity": "negative"}, {"category": "FACILITIES#GENERAL", "polarity": "negative"}, {"category": "FACILITIES#COMFORT", "polarity": "negative"}, {"category": "FACILITIES#DESIGN_FEATURES", "polarity": "negative"}, {"category": "FACILITIES#MISCELLANEOUS", "polarity": "negati

In [ ]:
save_path = "/content/drive/MyDrive/FYP/absa/rag_results/train_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)
